[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xylem-lab/GEO-ANOM-dashboard/blob/main/notebooks/07_review_detections.ipynb)

# 07 — Check detections by eye: poultry house or not?

**Question.** Of the houses mapped in notebook 05 that no one has verified, which are real poultry houses? Two kinds need a person to look:
- **Eastern Shore, not in the USGS tracing** (~280): built after 2016/17 (real), or a mistake. Settles the true precision of the counties notebook 06 uses (now 89% to ~100%).
- **Outside the traced area** (western counties, part of Cecil): every detection. The first ones looked like warehouses and an airport taxiway. Confirmed houses can then be used in notebook 06.

Mixed in, blind: **40 detections that sit on a hand-traced house** (known real). If those don't come back as *poultry house* nearly every time, the labels need a second look.

**How.** One photo at a time from the 2023 NAIP images the run used: 500 m around the detection (left) and a close-up (right), the outline in yellow. You see nothing else: no county, no group, no model score. Click *poultry house / not poultry / unsure*; *← back* lets you change the previous answer. Labels are saved after every click to `Task1_results/review/labels_<REVIEWER>.csv` on the shared drive. Stop any time and run again later; it continues where you left off. A second person can label the same set under their own name, and section 3 then reports agreement between reviewers.

**What counts as a poultry house:** a long, narrow, single-storey barn (typically 120–180 m × 12–18 m), usually in a row of similar barns on a farm, often with feed bins, fans at one end, and a litter shed nearby. If you can't tell from the photo, choose *unsure*; don't guess.

**Outputs.** `Task1_results/review/` (labels per reviewer, summary tables). Notebook 06 picks up confirmed houses outside the traced area automatically.
**Runtime.** About 3–5 s per photo on a home connection, hidden by prefetching; 500–600 photos ≈ 1–2 hours of clicking, in as many sittings as you like. Faster in Colab.

In [ ]:
# ── Setup: the same cell works in Google Colab and locally ──────────────────────
import os, sys, json, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "main"   # branch, tag or commit to run; the exact commit is recorded in run_meta.json

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                     # results go to the shared drive's Task1_results
    ROOT = Path("/content/GEO-ANOM-dashboard")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/xylem-lab/GEO-ANOM-dashboard", str(ROOT)], check=True)
    # always bring the copy up to date (a reused Colab session keeps an old copy)
    head = lambda: subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout
    before = head()
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=True)
    remote = f"origin/{CODE_VERSION}"
    target = remote if subprocess.run(["git", "-C", str(ROOT), "rev-parse", "-q", "--verify", remote],
                                      capture_output=True).returncode == 0 else CODE_VERSION
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", target], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
    if head() != before and "geo_anom" in sys.modules:   # new code, but the old version is already loaded
        raise RuntimeError("Code updated. Runtime -> Restart session, then Run all again.")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
# reload edited geo_anom/ code automatically, without restarting the kernel
if "google.colab" not in sys.modules:   # Colab's IPython can't load autoreload on Python 3.12+ (no `imp`)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

from geo_anom.task1 import env, tiles, region, viz
from geo_anom.task1 import county as C
from geo_anom.task1.kmz import write_kmz
from geo_anom.task1.pipeline import run_pipeline
from geo_anom.task1.evaluate import evaluate_run, load_ground_truth
from geo_anom.task1 import review as R
from geo_anom.task1.pipeline import git_commit
print("running in", "Colab" if IN_COLAB else "local", "| results ->", env.results_root())

## Settings

In [ ]:
REVIEWER = os.environ.get("GEO_ANOM_REVIEWER", "")   # your name or initials, e.g. "UA" -- one label file per reviewer
ORDER = "eastern_shore_first"   # or "mixed"; controls are always mixed in with the Eastern Shore set
N_CONTROLS = 40
COUNTIES = [n for n in C.all_counties_in_order() if C.is_complete(n)]
REVIEW_DIR = env.results_root() / "review"
print(len(COUNTIES), "finished counties | labels ->", REVIEW_DIR)

## 1. What there is to check

In [ ]:
queue = R.build_queue(COUNTIES, n_controls=N_CONTROLS)
if ORDER == "eastern_shore_first":
    queue = pd.concat([queue[queue.group != "unverified"], queue[queue.group == "unverified"]], ignore_index=True)
display(queue.pivot_table(index="county", columns="group", values="key", aggfunc="count", fill_value=0)
        .reindex(columns=R.GROUPS, fill_value=0))
print(f"{len(queue)} photos in the queue")

## 2. Review

Set `REVIEWER` above first. If the photo is slow, wait a second: the next few load in the background.

In [ ]:
if REVIEWER.strip():
    store = R.LabelStore(REVIEW_DIR / f"labels_{REVIEWER.strip()}.csv", REVIEWER, code=git_commit())
    print(f"{len(store.done() & set(queue.key))} of {len(queue)} already labelled by {REVIEWER}")
    ui = R.review_ui(queue, store)
else:
    print("Set REVIEWER in the Settings cell (your name or initials), run it, then run this cell.")

## 3. Results so far

Run this any time, also halfway through.
- **Precision per Eastern Shore county:** *low* counts unsure and not-yet-checked as wrong, *high* counts them as right. The two meet when everything is labelled.
- **Controls:** share of known-real houses the reviewer called a poultry house. Below ~95% means the labels are too strict and should be looked at.
- **Outside the traced area:** per county, how many detections are real.
- **Between reviewers** (when there are two or more): same-label share and Cohen's kappa (1 = perfect, above ~0.8 is strong).

In [ ]:
labels = R.load_labels(REVIEW_DIR)
summaries = {n: json.loads((C.county_out(n) / "county_summary.json").read_text()) for n in COUNTIES}
res = R.summarize(queue, labels, summaries)
print("Labels so far:", len(labels), "from", labels.reviewer.nunique() if len(labels) else 0, "reviewer(s)")
display(Markdown("**Eastern Shore precision**"))
show = res["precision"].copy()
for c in ["precision (traced only)", "precision low", "precision high"]:
    if c in show: show[c] = show[c].map("{:.1%}".format)
display(show)
display(Markdown(f"**Controls** (known real): {res['control']}"))
display(Markdown("**Outside the traced area**"))
display(res["unverified"])
if res["agreement"] is not None:
    display(Markdown("**Between reviewers**")); display(res["agreement"].round(3))

## 4. What the rejected ones look like

Up to 12 detections labelled *not poultry*: the failure modes the detector needs a filter for (and the negative examples for a future classifier).

In [ ]:
from IPython.display import Image as IPImage
bad = res["by_key"][res["by_key"].label == "not poultry"]
if not len(bad):
    print("Nothing labelled 'not poultry' yet.")
for _, r in queue[queue.key.isin(bad.key)].head(12).iterrows():
    display(Markdown(f"`{r.county}` · {r.length_m:.0f} × {r.width_m:.0f} m · model score {r.mean_prob:.2f}"))
    display(IPImage(R.render(r, R.chip(r)), width=780))

## 5. Save the summary

In [ ]:
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
res["by_key"].to_csv(REVIEW_DIR / "consensus_by_detection.csv", index=False)
if len(res["precision"]): res["precision"].to_csv(REVIEW_DIR / "eastern_shore_precision.csv")
if len(res["unverified"]): res["unverified"].to_csv(REVIEW_DIR / "unverified_by_county.csv")
(REVIEW_DIR / "summary_meta.json").write_text(json.dumps({
    "run_at": pd.Timestamp.now().isoformat(timespec="seconds"), "git_commit": git_commit(),
    "counties": COUNTIES, "queue": int(len(queue)), "labels": int(len(labels)),
    "reviewers": sorted(labels.reviewer.unique().tolist()) if len(labels) else [],
    "control": res["control"], "confirmed": len(R.confirmed_keys(labels))}, indent=2, default=str))
print("saved to", REVIEW_DIR)

## What this shows

*(Write after labelling: the Eastern Shore precision once every detection is checked, how many of the "not traced" detections were new houses, what share of detections outside the traced area are real, the failure modes in section 4, control agreement, and agreement between reviewers if there were two.)*